In [19]:
import confnotebook

In [20]:
from pathlib import Path

source = Path("../examples/RPA-6542/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] 18892339
[17] 18892958
[18] 18892969
[19] 18897098
[20] 7-1
[21] [Untitled]_23-48


In [21]:
IDX_FILE = 21

In [22]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

In [23]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-07-19 22:56:43.547 | INFO     | vision_core.pipelines.build_document:build:105 - Обработка страницы 0 с dpi 200...
2026-07-19 22:56:43.599 | INFO     | vision_core.pipelines.build_document:_process_page:187 - Коррекция ориентации и наклона...
2026-07-19 22:56:43.605 | DEBUG    | vision_core.preprocessor.image_orientation:process:44 - Ориентация страницы: 0° с точностью 0.9198
2026-07-19 22:56:43.618 | DEBUG    | vision_core.preprocessor.image_orientation:compute_deskew_angle:116 - Углы наклона страницы: 0.2128°
2026-07-19 22:56:43.702 | DEBUG    | vision_core.debug_image_observer:on_debug_image:56 - Debug image saved: 3_aligned -> ../examples/output/[Untitled]_23-48/3_aligned/page_000.png
2026-07-19 22:56:43.703 | INFO   

In [24]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-07-19 22:56:44.858 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 666 символов из 1 страниц
2026-07-19 22:56:44.859 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'по данным продавца' и 'кредит' (ratio=0.11)
2026-07-19 22:56:44.859 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'по данным продавца' и 'дебет' (ratio=0.06)
2026-07-19 22:56:44.860 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'по данным покупателя' и 'кредит' (ratio=0.10)
2026-07-19 22:56:44.860 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'по данным покупателя' и 'дебет' (ratio=0.10)
2026-07-19 22:56:44.861 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'дебет' и 'кредит' (ratio=0.33)
2026-07-1

In [25]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НА 01.01.2026', value=0.0, date='01.01.2026', row_reference=RowReference(id_table='0', id_row='3', id_col=1, buyer_col=3)), LedgerEntry(record='НЕУСТОЙКА (ШТРАФЫ, ПЕНИ)', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='4', id_col=1, buyer_col=3)), LedgerEntry(record='ПЕРЕДАНО МОЩНОСТИ:', value=8960271.77, date=None, row_reference=RowReference(id_table='0', id_row='5', id_col=1, buyer_col=3)), LedgerEntry(record='НЕУСТОЙКА (ШТРАФЫ, ПЕНИ)', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='6', id_col=1, buyer_col=3)), LedgerEntry(record='ОПЛАЧЕНО МОЩНОСТИ', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='7', id_col=1, buyer_col=3)), LedgerEntry(record='НЕУСТОЙКА (ШТРАФЫ, ПЕНИ)', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='8', id_col=1, buyer_col=3)), LedgerEntry(record='САЛЬДО НА 31.03.2026', value=163627.64, date='31.03.2026', row_reference=RowReference(id_table='0',

In [26]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = f"""
            По данным покупателя {data.buyer}
            По данным продавца {data.seller}
            В период: {data.period.start} - {data.period.end}
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-07-19 22:56:44.901 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf
2026-07-19 22:56:44.958 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-07-19 22:56:44.960 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R3:C3 значение=0.0
2026-07-19 22:56:44.961 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_font:229 - загружаем шрифт из /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf для размера 25
2026-07-19 22:56:44.962 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R4:C3 значение=0.0
2026-07-19 22:56:44.962 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R5:C3 значение=8960271.77
20

In [27]:
out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
Path(out_path).write_bytes(filled_pdf)
print(out_path)

../examples/output/[Untitled]_23-48_filled.pdf
